# Team 2 — CNN NILM (Kettle ON) — Upload-to-Colab Version

This version avoids Drive path issues by loading CSVs from `/content/` after you upload them into the Colab session.

**Train:** House 2

**Test:** House 1

**Label:** `kettle_on` derived from `kettle_watts >= THRESHOLD_W` (not `>0` to avoid noise).

## 0) Upload files into Colab

In the Colab left sidebar: **Files → Upload**

Upload these two CSVs (exact names):
- `House_2_kettle_analysis - House_2_kettle_analysis.csv`
- `House_1_kettle_analysis - House_1_kettle_analysis.csv`

These are the filenames visible in your Google Drive after export/upload. [Source](https://www.genspark.ai/api/files/s/4ll0gfS1)

In [2]:
import os, json
from pathlib import Path
import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import classification_report, confusion_matrix


In [3]:
!pip -q install numpy pandas scikit-learn tensorflow kagglehub

In [4]:
from google.colab import files
uploaded = files.upload()
print(uploaded.keys())


Saving House_1_kettle_analysis - House_1_kettle_analysis.csv to House_1_kettle_analysis - House_1_kettle_analysis.csv
dict_keys(['House_1_kettle_analysis - House_1_kettle_analysis.csv'])


In [5]:
from google.colab import files

uploaded = files.upload()
print("Uploaded files:", list(uploaded.keys()))


Saving House_2_kettle_analysis - House_2_kettle_analysis.csv to House_2_kettle_analysis - House_2_kettle_analysis.csv
Uploaded files: ['House_2_kettle_analysis - House_2_kettle_analysis.csv']


In [9]:
# After uploading via Colab UI (Files → Upload), run this to confirm the files are there

import glob, os

print("Files currently in /content:")
print("\n".join(sorted(os.listdir("/content"))))

h2 = sorted(glob.glob("/content/*House_2*kettle*analysis*.csv"))
h1 = sorted(glob.glob("/content/*House_1*kettle*analysis*.csv"))

print("\nHouse 2 CSV candidates:", h2)
print("House 1 CSV candidates:", h1)

if not h2 or not h1:
    raise FileNotFoundError("Upload both CSVs first (Files → Upload). Then rerun this cell.")

HOUSE2_KETTLE_CSV = h2[0]
HOUSE1_KETTLE_CSV = h1[0]

print("\nUsing:")
print("HOUSE2_KETTLE_CSV =", HOUSE2_KETTLE_CSV)
print("HOUSE1_KETTLE_CSV =", HOUSE1_KETTLE_CSV)
print("exists2:", os.path.exists(HOUSE2_KETTLE_CSV))
print("exists1:", os.path.exists(HOUSE1_KETTLE_CSV))


Files currently in /content:
.config
House_1_kettle_analysis - House_1_kettle_analysis.csv
House_2_kettle_analysis - House_2_kettle_analysis.csv
sample_data

House 2 CSV candidates: ['/content/House_2_kettle_analysis - House_2_kettle_analysis.csv']
House 1 CSV candidates: ['/content/House_1_kettle_analysis - House_1_kettle_analysis.csv']

Using:
HOUSE2_KETTLE_CSV = /content/House_2_kettle_analysis - House_2_kettle_analysis.csv
HOUSE1_KETTLE_CSV = /content/House_1_kettle_analysis - House_1_kettle_analysis.csv
exists2: True
exists1: True


## 1) Load House 2 (train) and House 1 (test) from `/content/`

In [7]:
HOUSE2_KETTLE_CSV = "/content/House_2_kettle_analysis - House_2_kettle_analysis.csv"  # train/val
HOUSE1_KETTLE_CSV = "/content/House_1_kettle_analysis - House_1_kettle_analysis.csv"  # test

print('House2 exists:', os.path.exists(HOUSE2_KETTLE_CSV), HOUSE2_KETTLE_CSV)
print('House1 exists:', os.path.exists(HOUSE1_KETTLE_CSV), HOUSE1_KETTLE_CSV)

train_df = pd.read_csv(HOUSE2_KETTLE_CSV)
test_df  = pd.read_csv(HOUSE1_KETTLE_CSV)

for df in (train_df, test_df):
    df['index'] = pd.to_datetime(df['index'], utc=True, errors='coerce')
    df.dropna(subset=['index','power_watts','kettle_watts'], inplace=True)
    df.sort_values('index', inplace=True)
    df.reset_index(drop=True, inplace=True)

print('Train df cols:', train_df.columns.tolist(), 'rows:', len(train_df))
print('Test  df cols:', test_df.columns.tolist(), 'rows:', len(test_df))
print(train_df.head())


House2 exists: True /content/House_2_kettle_analysis - House_2_kettle_analysis.csv
House1 exists: True /content/House_1_kettle_analysis - House_1_kettle_analysis.csv
Train df cols: ['index', 'power_watts', 'kettle_watts'] rows: 5630
Test  df cols: ['index', 'power_watts', 'kettle_watts'] rows: 5630
                      index  power_watts  kettle_watts
0 2013-02-17 16:00:00+00:00   638.716736           0.0
1 2013-02-17 17:00:00+00:00   573.896912           0.0
2 2013-02-17 18:00:00+00:00   574.044983           0.0
3 2013-02-17 19:00:00+00:00  1860.316528           0.0
4 2013-02-17 20:00:00+00:00   424.234070           0.0


## 2) Create Boolean label `kettle_on` from a threshold

In [12]:
THRESHOLD_W = 500  # start here; tune based on counts/top events

train_df["kettle_on"] = (train_df["kettle_watts"] >= THRESHOLD_W).astype(int)
test_df["kettle_on"]  = (test_df["kettle_watts"]  >= THRESHOLD_W).astype(int)

print("Threshold:", THRESHOLD_W)
print("\nTrain kettle_on counts:")
print(train_df["kettle_on"].value_counts(dropna=False))

print("\nTest kettle_on counts:")
print(test_df["kettle_on"].value_counts(dropna=False))

print("\nTop kettle_watts (train):")
print(
    train_df.sort_values("kettle_watts", ascending=False)
            .head(10)[["index", "kettle_watts", "power_watts", "kettle_on"]]
)


Threshold: 500

Train kettle_on counts:
kettle_on
0    5614
1      16
Name: count, dtype: int64

Test kettle_on counts:
kettle_on
0    5630
Name: count, dtype: int64

Top kettle_watts (train):
                         index  kettle_watts  power_watts  kettle_on
2209 2013-05-20 17:00:00+00:00    1179.58620     0.000000          1
2211 2013-05-20 19:00:00+00:00    1179.58620     0.000000          1
2208 2013-05-20 16:00:00+00:00    1179.58620  1382.116333          1
2212 2013-05-20 20:00:00+00:00    1179.58620     0.000000          1
2210 2013-05-20 18:00:00+00:00    1179.58620     0.000000          1
4907 2013-09-10 03:00:00+00:00     503.36603     0.000000          1
4905 2013-09-10 01:00:00+00:00     503.36603     0.000000          1
4906 2013-09-10 02:00:00+00:00     503.36603     0.000000          1
4897 2013-09-09 17:00:00+00:00     503.36603   862.255798          1
4898 2013-09-09 18:00:00+00:00     503.36603     0.000000          1


## 3) Windowing parameters based on sampling interval

In [14]:
train_delta = train_df["index"].diff().value_counts().head(5)
print("Most common train deltas:\n", train_delta)

most_common = train_df["index"].diff().mode().iloc[0] if len(train_df) > 2 else pd.Timedelta(hours=1)

if most_common <= pd.Timedelta(minutes=2):
    WINDOW = 256
    STRIDE = 16
else:
    WINDOW = 24
    STRIDE = 1

print("Using WINDOW=", WINDOW, "STRIDE=", STRIDE)


Most common train deltas:
 index
0 days 01:00:00    5629
Name: count, dtype: int64
Using WINDOW= 24 STRIDE= 1


## 4) Build windows: X = mains `power_watts`, y = `kettle_on` (end-of-window label)

In [15]:
def make_windows(df, window, stride):
    x = df['power_watts'].astype('float32').to_numpy()
    y = df['kettle_on'].astype('int32').to_numpy()
    X, Y = [], []
    for end in range(window, len(df), stride):
        start = end - window
        X.append(x[start:end])
        Y.append(y[end-1])
    X = np.array(X, dtype='float32')[..., None]
    Y = np.array(Y, dtype='int32')
    return X, Y

X_all, y_all = make_windows(train_df, WINDOW, STRIDE)
X_test, y_test = make_windows(test_df, WINDOW, STRIDE)

print('House2 windows:', X_all.shape, y_all.shape)
print('House1 windows:', X_test.shape, y_test.shape)
print('Train positive rate:', y_all.mean(), 'Test positive rate:', y_test.mean())


House2 windows: (5606, 24, 1) (5606,)
House1 windows: (5606, 24, 1) (5606,)
Train positive rate: 0.0028540849090260435 Test positive rate: 0.0


## 5) Train/Val split (House 2 only) + normalization (fit on House 2 train only)

In [16]:
VAL_FRAC = 0.15
N = len(X_all)
split = int(N * (1 - VAL_FRAC))

X_train, y_train = X_all[:split], y_all[:split]
X_val, y_val     = X_all[split:], y_all[split:]

mu = X_train.mean()
sd = X_train.std() + 1e-6

X_train_n = (X_train - mu) / sd
X_val_n   = (X_val   - mu) / sd
X_test_n  = (X_test  - mu) / sd

print('Train/Val/Test shapes:', X_train_n.shape, X_val_n.shape, X_test_n.shape)
print('Norm mu/sd:', float(mu), float(sd))


Train/Val/Test shapes: (4765, 24, 1) (841, 24, 1) (5606, 24, 1)
Norm mu/sd: 291.8094177246094 288.5320739746094


## 6) Class weights (handle imbalance)

In [17]:
neg = int((y_train == 0).sum())
pos = int((y_train == 1).sum())

if pos == 0:
    raise ValueError('No positive examples. Lower THRESHOLD_W or adjust labeling/windowing.')

w0 = 0.5 * (len(y_train) / neg)
w1 = 0.5 * (len(y_train) / pos)
class_weight = {0: w0, 1: w1}

print('neg:', neg, 'pos:', pos)
print('class_weight:', class_weight)


neg: 4760 pos: 5
class_weight: {0: 0.5005252100840336, 1: 476.5}


## 7) CNN model + callbacks (logging/checkpoints)

In [18]:
def make_cnn(window):
    inp = keras.Input(shape=(window, 1))
    x = layers.Conv1D(32, 7, padding='same', activation='relu')(inp)
    x = layers.MaxPool1D(2)(x)
    x = layers.Conv1D(64, 5, padding='same', activation='relu')(x)
    x = layers.MaxPool1D(2)(x)
    x = layers.Conv1D(128, 3, padding='same', activation='relu')(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='relu')(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    model = keras.Model(inp, out)
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss='binary_crossentropy',
        metrics=[
            keras.metrics.BinaryAccuracy(name='acc'),
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall'),
            keras.metrics.AUC(name='auc'),
        ],
    )
    return model

model = make_cnn(WINDOW)
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 24, 1)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 24, 32)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 12, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 12, 64)         │        10,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 6, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 6, 128)         │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 43,585 (170.25 KB)

 Trainable params: 43,585 (170.25 KB)

 Non-trainable params: 0 (0.00 B)

In [19]:
Path('reports').mkdir(exist_ok=True)
Path('models').mkdir(exist_ok=True)

run_name = f'cnn_house2_to_house1_kettle_on_thr{THRESHOLD_W}'

callbacks = [
    keras.callbacks.CSVLogger(f'reports/{run_name}_history.csv', append=False),
    keras.callbacks.ModelCheckpoint(
        filepath=f'models/{run_name}.keras',
        monitor='val_auc',
        mode='max',
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_auc',
        mode='max',
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
]

history = model.fit(
    X_train_n, y_train,
    validation_data=(X_val_n, y_val),
    epochs=30,
    batch_size=128,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1,
)


Epoch 1/30
37/38 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - acc: 0.5450 - auc: 0.6146 - loss: 1.0470 - precision: 0.0017 - recall: 0.3838
Epoch 1: val_auc improved from -inf to 0.88499, saving model to models/cnn_house2_to_house1_kettle_on_thr500.keras
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step - acc: 0.5553 - auc: 0.6215 - loss: 1.0242 - precision: 0.0017 - recall: 0.3846 - val_acc: 0.9869 - val_auc: 0.8850 - val_loss: 0.1911 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 2/30
36/38 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - acc: 0.9876 - auc: 0.3869 - loss: 0.4661 - precision: 0.0011 - recall: 0.0111        
Epoch 2: val_auc improved from 0.88499 to 0.99858, saving model to models/cnn_house2_to_house1_kettle_on_thr500.keras
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - acc: 0.9872 - auc: 0.4210 - loss: 0.5023 - precision: 0.0020 - recall: 0.0256 - val_acc: 0.8359 - val_auc: 0.9986 - val_loss: 0.2599 - val_precision: 0.0738 - val_recall: 1.0000
Epoch 3/30
37/38 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/ste

## 8) Evaluate on House 1 (report + confusion matrix)

In [20]:
y_prob = model.predict(X_test_n).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print('Classification report:')
print(classification_report(y_test, y_pred, digits=4))

print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))

config = {
    'threshold_w': int(THRESHOLD_W),
    'window': int(WINDOW),
    'stride': int(STRIDE),
    'val_frac': float(VAL_FRAC),
    'norm_mu': float(mu),
    'norm_sd': float(sd),
}
with open(f'reports/{run_name}_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('Saved:', f'reports/{run_name}_history.csv', f'models/{run_name}.keras', f'reports/{run_name}_config.json')


176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Classification report:
              precision    recall  f1-score   support

           0     1.0000    0.8514    0.9197      5606
           1     0.0000    0.0000    0.0000         0

    accuracy                         0.8514      5606
   macro avg     0.5000    0.4257    0.4599      5606
weighted avg     1.0000    0.8514    0.9197      5606

Confusion matrix:
[[4773  833]
 [   0    0]]
Saved: reports/cnn_house2_to_house1_kettle_on_thr500_history.csv models/cnn_house2_to_house1_kettle_on_thr500.keras reports/cnn_house2_to_house1_kettle_on_thr500_config.json


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
